# CM4044 MP2-30 · How noisy is toxicity? Label noise and ensemble robustness on Tox21

This notebook runs the whole project on **Google Colab** (free T4 GPU). Every number in the slides is produced by the cells below.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

| Step | Cell | Approx. time on Colab T4 |
|---|---|---|
| 0 | Setup (Drive, clone, install) | 2–3 min |
| 1 | Curation + natural-noise analysis (Task 1) | 1 min |
| 2–3 | Noise × defence grid: MLP + RF | ~1.5 h |
| 2–3 | Noise × defence grid: GNN | ~2–3 h |
| 4 | Real-suspect validation (Task 4) | ~10 min |
| S | Stretch: multi-task noise interaction | ~30 min |

Results are written to **Google Drive** after every single run, so a Colab disconnect never loses work: just re-run the cell and finished runs are skipped. The grid can also be **split across the three group members** (each runs a different `--seeds` value on their own Colab account in the same shared Drive folder, or copies their `results/runs/*.json` files together afterwards).

## 0 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - switch the runtime to T4 GPU"

Mount Google Drive so that results and checkpoints survive a disconnect. The repository is cloned **into Drive** once; later sessions just `git pull`.

If the GitHub repo is **private**, create a fine-grained personal access token (GitHub → Settings → Developer settings → Tokens, read-only *Contents* access on this repo) and paste it when asked. Alternatively upload the repo ZIP to the Drive folder and skip the clone.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, getpass, subprocess
REPO = 'jev-hxr/CM4044_Project_2'
BRANCH = 'claude/cm4044-mini-project-2-mgdn0t'   # change to 'main' once merged
WORKDIR = '/content/drive/MyDrive/CM4044_MP2'

if not os.path.exists(f'{WORKDIR}/noisytox'):
    url = f'https://github.com/{REPO}.git'
    r = subprocess.run(['git', 'clone', '-b', BRANCH, url, WORKDIR], capture_output=True, text=True)
    if r.returncode != 0:   # private repo -> use a token
        token = getpass.getpass('GitHub token (input hidden): ')
        subprocess.run(['git', 'clone', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', WORKDIR], check=True)
else:
    subprocess.run(['git', '-C', WORKDIR, 'pull', '--ff-only'])
os.chdir(WORKDIR)
!git log --oneline -1

In [ ]:
# Colab already ships torch, scikit-learn, pandas, matplotlib, scipy.
!pip install -q -r requirements-colab.txt
import torch, torch_geometric, rdkit, cleanlab, sklearn
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| PyG', torch_geometric.__version__,
      '| RDKit', rdkit.__version__, '| cleanlab', cleanlab.__version__, '| sklearn', sklearn.__version__)
!python -m noisytox.versions

## 1 · Data curation and natural noise (Task 1)
Standardise → strip salts → neutralise → merge duplicates (replicate conflicts are logged) → scaffold split. Then characterise the *natural* noise: class imbalance per assay, replicate disagreement and cross-assay disagreement (AR vs AR-LBD, ER vs ER-LBD).

In [ ]:
!python -m noisytox.curate

In [ ]:
!python -m noisytox.natural_noise
from IPython.display import Image, display
display(Image('figures/fig_imbalance.png', width=650)); display(Image('figures/fig_assay_kappa.png', width=550))

## 2–3 · Synthetic noise × defences grid (Tasks 2 and 3)
Conditions: clean, symmetric (`sym`) and class-conditional prior-preserving (`cc`) noise at 5 / 10 / 20 %, seeds 0–2.
Models: RF (classical baseline), MLP on ECFP4, GIN graph network. Defences: `none` (baseline), `ls` label smoothing, `fc` forward loss correction (estimated T), `fc_oracle` (true T, upper bound), `cl` confident learning, `ed` ensemble-disagreement filter.

Each run prints one line and saves `results/runs/<model>_<defence>_<condition>_s<seed>.json`. **Re-running a cell resumes where it stopped.**

*Splitting the work across the group:* add e.g. `--seeds 1` to every grid command on member B's account, `--seeds 2` on member C's.

In [ ]:
# MLP + RF (fast). --save-models stores seed-0 checkpoints in models/checkpoints/
!python -m noisytox.run_grid --models mlp rf --save-models

In [ ]:
# GNN (slowest part: ~1 min per run on a T4)
!python -m noisytox.run_grid --models gnn --save-models

In [ ]:
!python -m noisytox.aggregate
display(Image('figures/fig_degradation_roc.png', width=900))
display(Image('figures/fig_degradation_pr.png', width=900))
display(Image('figures/fig_delta_heatmap.png', width=800))
display(Image('figures/fig_detection.png', width=900))

## 4 · Do the detectors flag *real* suspect labels? (Task 4)
Detectors are run on the natural labels of all 7.5k molecules (out-of-fold, so a molecule's partner-assay label is never seen). Suspect labels (related assays disagree) are compared with concordant labels, **within label class** to remove the active/inactive confound.

In [ ]:
!python -m noisytox.validate_suspects
display(Image('figures/fig_suspects.png', width=850))

## S · Stretch: multi-task noise interaction
20 % symmetric noise is injected into the 7 nuclear-receptor (NR) assays only; the 5 stress-response (SR) assays stay clean. Does noise in the NR heads hurt the shared representation used by the SR heads, and does cleaning the NR labels help the SR assays?

In [ ]:
!python -m noisytox.interaction
display(Image('figures/fig_interaction.png', width=850))

## Deliverables: model cards and the submission zip

In [ ]:
!python -m noisytox.model_cards
!zip -qr /content/CM4044_MP2-30_deliverables.zip README.md requirements*.txt environment.yml noisytox notebooks data/raw data/tox21_curated.csv data/curation_log.md data/duplicate_conflicts.csv results/*.csv results/*.md results/runs results/detection results/interaction figures models -x '*/__pycache__/*'
!ls -lh /content/CM4044_MP2-30_deliverables.zip
from google.colab import files
files.download('/content/CM4044_MP2-30_deliverables.zip')